In [15]:
import numpy as np
from ctypes import CDLL, POINTER, c_double
from pathlib import Path
from scipy.integrate import solve_ivp

from scipy.optimize import brentq

In [16]:
lib = CDLL("./libequations.dylib")


# Metric
Matrix4 = (c_double * 4) * 4
lib.gmetric.argtypes = [
    c_double,
    c_double,
    POINTER(Matrix4),
    POINTER(c_double)
]
lib.gmetric.restype = None


# Equations
lib.equations.argtypes = [
    POINTER(c_double),  # varst
    POINTER(c_double),  # diffs
    c_double,           # impact_par
    POINTER(c_double),  # parbh
]
lib.equations.restype = None









In [17]:
# 3-qadam. haqiqiy wrapper yozamiz
def metric_components(
    r,
    theta,
    parbh
):

    g = Matrix4()

    parbh_array = np.ascontiguousarray(
        parbh,
        dtype=np.float64
    )

    lib.gmetric(
        c_double(r),
        c_double(theta),
        g,
        parbh_array.ctypes.data_as(
            POINTER(c_double)
        )
    )

    return np.array(g)



def equations(
    varst: np.ndarray,
    impact_par: float,
    parbh: np.ndarray,
) -> np.ndarray:

    varst = np.ascontiguousarray(varst, dtype=np.float64)
    parbh = np.ascontiguousarray(parbh, dtype=np.float64)

    diffs = np.zeros_like(varst, dtype=np.float64)

    lib.equations(
        varst.ctypes.data_as(POINTER(c_double)),
        diffs.ctypes.data_as(POINTER(c_double)),
        c_double(impact_par),
        parbh.ctypes.data_as(POINTER(c_double)),
    )

    return diffs

In [18]:
# Test   #########





# tekshirib olamiz
g = metric_components(
    100,
    1.2,
    [1., 0.5]
)

print(g)


varst = np.array([1.0, 2.0, 3.0, 4.0, 5.0])

parbh = np.array([1.0, 0.1])

impact_par = 0.11

diffs = equations(
    varst,
    impact_par,
    parbh
)

print("//////")
print(diffs)

[[-9.80025000e-01  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  1.02038213e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+04  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  8.68696858e+03]]
//////
[  3.           4.         -23.85449001 -24.00669752   0.13303955]


In [19]:
def initial_photon_geometry(
    alpha,
    beta,
    d,
    inclination
):
    """
    Construct the initial photon position and spatial momentum
    from the observer screen coordinates (alpha,beta).
    """

    # ---------------------------------------------------------
    # Initial position
    # ---------------------------------------------------------

    r0 = np.sqrt(
        d**2 +
        alpha**2 +
        beta**2
    )

    theta0 = np.arccos(
        (
            d*np.cos(inclination)
            +
            beta*np.sin(inclination)
        ) / r0
    )

    phi0 = np.arctan2(
        alpha,
        d*np.sin(inclination)
        -
        beta*np.cos(inclination)
    )

    # ---------------------------------------------------------
    # Spatial momentum
    # ---------------------------------------------------------

    kr = -d / r0

    numerator = (
        -np.cos(inclination)
        +
        (
            d / r0**2
        )
        *
        (
            d*np.cos(inclination)
            +
            beta*np.sin(inclination)
        )
    )

    denominator = np.sqrt(
        r0**2
        -
        (
            d*np.cos(inclination)
            +
            beta*np.sin(inclination)
        )**2
    )

    ktheta = numerator / denominator

    kphi = (
        -alpha*np.sin(inclination)
        /
        (
            alpha**2
            +
            (
                d*np.sin(inclination)
                -
                beta*np.cos(inclination)
            )**2
        )
    )

    return (
        r0,
        theta0,
        phi0,
        kr,
        ktheta,
        kphi
    )

In [20]:
# Test   #########

### test check
d = 10000
inclination = np.deg2rad(60)

r0, theta0, phi0, kr, ktheta, kphi = initial_photon_geometry(
    10,
    5,
    d,
    inclination
)

print(r0)
print(theta0)
print(phi0)
print(kr)
print(ktheta)
print(kphi)

10000.006249998047
1.0466978402465152
0.0011550334543208862
-0.999999375000586
4.994215255987951e-08
-1.155365952454481e-07


In [21]:
# 3-qadam umumiy kerr uchun yozamiz photon motion va kt ni aniqlaymiz kv tenglama qilib
def solve_null_condition(
    g,
    kr,
    ktheta,
    kphi
):
    """
    Compute k^t from the null condition

        g_{μν} k^μ k^ν = 0

    Works for both static and stationary metrics.
    """

    gtt = g[0, 0]
    gtp = g[0, 3]
    grr = g[1, 1]
    gthth = g[2, 2]
    gpp = g[3, 3]

    A = gtt

    B = 2.0 * gtp * kphi

    C = (
        grr * kr**2
        +
        gthth * ktheta**2
        +
        gpp * kphi**2
    )

    discriminant = B**2 - 4.0*A*C

    if discriminant < 0:
        raise ValueError(
            "Negative discriminant in null condition."
        )

    sqrtD = np.sqrt(discriminant)

    kt1 = (-B + sqrtD)/(2*A)

    kt2 = (-B - sqrtD)/(2*A)

    return kt1, kt2

In [22]:
# Test   #########



# test qilamiz

g = metric_components(
    100,
    1.2,
    [1., 0.5]
)
kt1, kt2 = solve_null_condition(
    g,
    kr,
    ktheta,
    kphi
)

print("kt1 =", kt1)
print("kt2 =", kt2)

kt1 = -1.0203814954546222
kt2 = 1.0203814954546222


In [23]:
# 4-qadam : Conserved quantites ray-tracing uchun E=-p_t va Lz = p_]phi
def conserved_quantities(
    g,
    kt,
    kphi
):
    """
    Compute conserved quantities

        E  = -p_t
        Lz =  p_phi

    for a stationary axisymmetric spacetime.
    """

    gtt = g[0, 0]
    gtp = g[0, 3]
    gpp = g[3, 3]

    # Conserved energy
    E = -(
        gtt * kt
        +
        gtp * kphi
    )

    # Conserved angular momentum
    Lz = (
        gpp * kphi
        +
        gtp * kt
    )

    impact_par = Lz / E

    return E, Lz, impact_par

In [24]:
# Test   #########



# tekshirib olamiz kattaliklarni
g = metric_components(
    100,
    1.2,
    [1., 0.5]
)

d = 10000
inclination = np.deg2rad(60)

r0, theta0, phi0, kr, ktheta, kphi = initial_photon_geometry(
    10,
    5,
    d,
    inclination
)


kt1, kt2 = solve_null_condition(
    g,
    kr,
    ktheta,
    kphi
)


kt2 = 1.0203814954546222

kt = kt2

E, Lz, impact_par = conserved_quantities(
    g,
    kt,
    kphi
)

print("E =", E)
print("Lz =", Lz)
print("impact =", impact_par)

E = 0.99999937506963
Lz = -0.0010036627724723701
impact = -0.0010036633996921098


In [25]:
# Endi oxirgi boshlang'ich shart funksiyasini yozamiz
# bu funksiya tepadagi initial_photon_state() ni to'liq almashtiradi
def initial_photon_state(
    alpha,
    beta,
    d,
    inclination,
    parbh
):
    """
    Construct normalized initial conditions for the geodesic solver.
    """

    # --------------------------------------------------
    # Geometry
    # --------------------------------------------------

    (
        r0,
        theta0,
        phi0,
        kr,
        ktheta,
        kphi
    ) = initial_photon_geometry(
        alpha,
        beta,
        d,
        inclination
    )

    # --------------------------------------------------
    # Metric
    # --------------------------------------------------

    g = metric_components(
        r0,
        theta0,
        parbh
    )

    # --------------------------------------------------
    # Null condition
    # --------------------------------------------------

    kt_plus, kt_minus = solve_null_condition(
        g,
        kr,
        ktheta,
        kphi
    )

    # Choose the future-directed solution
    candidates = [kt_plus, kt_minus]
    positive = [k for k in candidates if k > 0]

    if len(positive) != 1:
        raise ValueError("Could not identify the future-directed solution.")

    kt = positive[0]

    # --------------------------------------------------
    # Conserved quantities
    # --------------------------------------------------

    E, Lz, impact_par = conserved_quantities(
        g,
        kt,
        kphi
    )

    # --------------------------------------------------
    # Normalize affine parameter
    # --------------------------------------------------

    kr /= E
    ktheta /= E

    varst0 = np.array([
        r0,
        theta0,
        kr,
        ktheta,
        phi0
    ])

    return varst0, impact_par

In [26]:
# Test   #########

alpha = 3.
beta = 0.001

d = 1000
inclination = np.deg2rad(60)

parbh = np.array([1., 0.5])

varst0, impact_par = initial_photon_state(
    alpha,
    beta,
    d,
    inclination,
    parbh
)

print(varst0)
print(impact_par)

[ 1.00000450e+03  1.04719915e+00 -9.99995509e-01 -4.19609252e-09
  3.46408976e-03]
-2.5980762347327664


In [30]:
# bitta photon uchun qaraymiz
def geodesic(lam, varst, impact_par, parbh):

    return equations(
        varst,
        impact_par,
        parbh
    )



# Horizon begin
def f_metric(r, mpar, qpar):

    return 1.0 - (
        2.0 * mpar / r
    ) * (
        1.0
        - np.tanh(
            qpar**2
            / (2.0 * mpar * r)
        )
    )


def rhorizon(parbh):
    mpar, qpar = parbh
    
    r_h = brentq(
        lambda r: f_metric(r, mpar, qpar),
        0.1,
        10.0
    )
    return r_h
    
# Horizon end



def hit_disk(lam, varst,impact_par, parbh):

    return varst[1] - np.pi / 2


hit_disk.terminal = True
hit_disk.direction = 1

def hit_horizon(lam, varst,impact_par, parbh):
    rh = rhorizon(parbh)
    rtol = 1.0e-6
    return varst[0] - rh*(1. + rtol)


hit_horizon.terminal = True
hit_horizon.direction = -1




def escapedOUT(lam, varst,impact_par, parbh):

    return varst[0] - 100000.0

def escapedIN(lam, varst,impact_par, parbh):
    isco = 1.0
    return varst[0] - isco

escapedOUT.terminal = True
escapedOUT.direction = 1

escapedIN.terminal = True
escapedIN.direction = -1

In [38]:
# Test ##########
alpha = 3.
beta = 0.001

d = 1000
inclination = np.deg2rad(60)

parbh = np.array([1., 0.5])

varst0, impact_par = initial_photon_state(
    alpha,
    beta,
    d,
    inclination,
    parbh
)

solution = solve_ivp(
    fun=geodesic,
    t_span=(0.0, 100000.0),
    y0=varst0,
    args=(impact_par, parbh),
    method="RK45",
    events=[
        hit_disk,
        hit_horizon,
        escapedOUT,
        escapedIN
    ],
    rtol=1e-8,
    atol=1e-8
)
print(solution.t_events[0])
print(solution.t_events[1])
print(solution.t_events[2])
print(solution.t_events[3])

[999.16744149]
[]
[]
[]
